In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

filename = '../Data/genres_original/pop/pop.00000.wav'
y, sr = librosa.load(filename)
print(f"Sample rate: {sr}")

librosa.display.waveshow(y, sr=sr)

In [ ]:
# Phát âm thanh
from IPython.display import Audio
Audio(data=y, rate=sr)

In [ ]:
### Truc quan hoa cac doan am thanh
audio_path = '../Data/genres_original/pop/pop.00000.wav'
y, sr = librosa.load(filename,sr=None) #sr=None de giu nguyen sample rate goc cua file am thanh
# dinh nghia do dai cua khung va buoc nhay
chunk_duration=4  
overlap_duration=2

# chuyen doi duration sang so luong mau
chunk_samples=chunk_duration*sr
overlap_samples=overlap_duration*sr

# tinh toan so luong khung co the tao ra tu doan am thanh
num_chunks=int(np.ceil((len(y)-chunk_samples)/(chunk_samples-overlap_samples)))+1

# tao cac khung am thanh
for i in range(num_chunks):
    # start va end index cua khung
    start=i*(chunk_samples-overlap_samples)
    end=start+chunk_samples

    # cat khung am thanh
    chunk=y[start:end]
    plt.figure(figsize=(4,2))
    librosa.display.waveshow(chunk, sr=sr)
    plt.tight_layout()
    plt.show()

In [ ]:
### Plotting MelSpectrogram Of Entire Audio
def plot_melspectrogram(y,sr):
    # Compute spectrogram
    spectrogram=librosa.feature.melspectrogram(y=y,sr=sr)
    # Convert to dB (log scale)
    spectrogram_db=librosa.power_to_db(spectrogram,ref=np.max)
    # Visualize the spectrogram
    plt.figure(figsize=(10,4))
    librosa.display.specshow(spectrogram_db,sr=sr,x_axis='time',y_axis='mel')
    plt.colorbar(format='%+2.0f dB')
    plt.title('MelSpectrogram')
    plt.xlabel('Time')
    plt.ylabel('Mel Frequency')
    plt.show()


In [ ]:
filename = '../Data/genres_original/pop/pop.00000.wav'
y, sr = librosa.load(filename,sr=44100)
plot_melspectrogram(y,sr)

In [ ]:
def plot_melspectrogram_chunks(y,sr):
    # dinh nghia do dai cua khung va buoc nhay
    chunk_duration=4  
    overlap_duration=2

    # chuyen doi duration sang so luong mau
    chunk_samples=chunk_duration*sr
    overlap_samples=overlap_duration*sr

    # tinh toan so luong khung co the tao ra tu doan am thanh
    num_chunks=int(np.ceil((len(y)-chunk_samples)/(chunk_samples-overlap_samples)))+1

    # tao cac khung am thanh
    for i in range(num_chunks):
        # start va end index cua khung
        start=i*(chunk_samples-overlap_samples)
        end=start+chunk_samples

        # cat khung am thanh
        chunk=y[start:end]
        plot_melspectrogram(chunk,sr)
        

In [ ]:
filename = '../Data/genres_original/pop/pop.00000.wav'
y, sr = librosa.load(filename,sr=44100)
plot_melspectrogram_chunks(y,sr)

In [ ]:
import os
import numpy as np
import librosa
from tensorflow.image import resize

# DATA PREPROCESSING - FINAL
data_dir = "../Data/genres_original" 
classes = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

def load_and_preprocess_audio(data_dir, classes, target_shape=(150, 150)):
    data = []
    labels = []

    for i_class, class_name in enumerate(classes):
        class_dir = os.path.join(data_dir, class_name)
        print(f"Đang xử lý thư mục: {class_name}...")
        
        for filename in os.listdir(class_dir):
            if filename.endswith('.wav'):
                file_path = os.path.join(class_dir, filename)
             
                try:
                    audio_data, sample_rate = librosa.load(file_path, sr=None)
                except Exception as e:
                    print(f"  -> [Cảnh báo] Bỏ qua file lỗi: {filename} ({e})")
                    continue 
                # -------------------------------

                chunk_duration = 4
                overlap_duration = 2

                chunk_samples = chunk_duration * sample_rate
                overlap_samples = overlap_duration * sample_rate

                # Tính toán số lượng khung
                num_chunks = int(np.ceil((len(audio_data) - chunk_samples) / (chunk_samples - overlap_samples))) + 1

                for i in range(num_chunks):
                    start = i * (chunk_samples - overlap_samples)
                    end = start + chunk_samples

                    # Đảm bảo chunk không vượt quá chiều dài của audio
                    chunk = audio_data[start:end]
                    
                    # Bỏ qua nếu chunk quá ngắn so với yêu cầu (thường ở cuối file)
                    if len(chunk) < chunk_samples:
                        continue

                    mel_spectrogram = librosa.feature.melspectrogram(y=chunk, sr=sample_rate)

                    # resize matrix mel spectrogram to 150x150
                    mel_spectrogram = resize(np.expand_dims(mel_spectrogram, axis=-1), target_shape)

                    # append mel spectrogram to data list and corresponding label to labels list
                    data.append(mel_spectrogram)
                    labels.append(i_class)

    return np.array(data), np.array(labels)



In [14]:
# Chạy hàm tiền xử lý
data, labels = load_and_preprocess_audio(data_dir, classes)
print(f"\nHoàn tất! Tổng số mẫu data thu được: {data.shape}")
print(f"Tổng số nhãn labels thu được: {labels.shape}")

Đang xử lý thư mục: blues...
Đang xử lý thư mục: classical...
Đang xử lý thư mục: country...
Đang xử lý thư mục: disco...
Đang xử lý thư mục: hiphop...
Đang xử lý thư mục: jazz...


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4164\1742655917.py:25: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_data, sample_rate = librosa.load(file_path, sr=None)


  -> [Cảnh báo] Bỏ qua file lỗi: jazz.00054.wav ()
Đang xử lý thư mục: metal...
Đang xử lý thư mục: pop...
Đang xử lý thư mục: reggae...
Đang xử lý thư mục: rock...

Hoàn tất! Tổng số mẫu data thu được: (13977, 150, 150, 1)
Tổng số nhãn labels thu được: (13977,)
